In [6]:
import pandas as pd

# Load the dataset
df = pd.read_csv(r"C:\Users\hp\OneDrive\Desktop\archive\train.csv", nrows = 1000)

# Print the first few rows of the dataset
df.head()

,Class Index,Title,Description
0,3,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli..."
1,3,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...
2,3,Oil and Economy Cloud Stocks' Outlook (Reuters),Reuters - Soaring crude prices plus worries\ab...
3,3,Iraq Halts Oil Exports from Main Southern Pipe...,Reuters - Authorities have halted oil export\f...
4,3,"Oil prices soar to all-time record, posing new...","AFP - Tearaway world oil prices, toppling reco..."


In [7]:
df.tail()

,Class Index,Title,Description
995,3,U.S. Stocks Rebound as Oil Prices Ease,NEW YORK (Reuters) - U.S. stocks rebounded on...
996,3,Dollar Rises Vs Euro After Asset Data,NEW YORK (Reuters) - The dollar gained agains...
997,4,Bikes Bring Internet to Indian Villagers (AP),"AP - For 12-year-old Anju Sharma, hope for a b..."
998,4,Celebrity Chefs Are Everywhere in Vegas,By ADAM GOLDMAN LAS VEGAS (AP) -- The waite...
999,4,Entertainment World Wary of Microsoft Technology,By GARY GENTILE LOS ANGELES (AP) -- CinemaN...


In [9]:
df.shape

(1000, 3)

In [10]:
df.isna().any()

Class Index    False
Title          False
Description    False
dtype: bool

In [11]:
df.dtypes

Class Index     int64
Title          object
Description    object
dtype: object

In [12]:
from transformers import BertModel, BertTokenizer
import torch

In [13]:
model = BertModel.from_pretrained('bert-base-uncased', output_hidden_states=True)
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

c:\Users\hp\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:144: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\hp\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better pe

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [15]:
def extract_features(text):
    # Tokenize the text
    input_ids = torch.tensor([tokenizer.encode(text, add_special_tokens=True)])
    # Get the hidden states for each token
    with torch.no_grad():
        outputs = model(input_ids)
        hidden_states = outputs[2]
    # Concatenate the last 4 hidden states
    token_vecs = []
    for layer in range(-4, 0):
        token_vecs.append(hidden_states[layer][0])
    # Calculate the mean of the last 4 hidden states
    features = []
    for token in token_vecs:
        features.append(torch.mean(token, dim=0))
    # Return the features as a tensor
    return torch.stack(features)

In [17]:
features = []
for i in range(len(df)):
    features.append(extract_features(df.iloc[i]["Description"]))
# Concatenate the features and convert to a numpy array
features = torch.cat(features).numpy()

In [18]:
features

array([[ 0.19557555, -0.30943725,  0.34961572, ..., -0.25235328,
         0.7551447 , -0.2660927 ],
       [ 0.22537155, -0.63355994,  0.4750447 , ..., -0.2490972 ,
         0.5154947 , -0.5139005 ],
       [ 0.24892987, -0.42132866,  0.352715  , ..., -0.3237999 ,
         0.47544193, -0.3927046 ],
       ...,
       [ 0.16037719, -0.08182271,  0.5958929 , ..., -0.01507375,
         0.31809986,  0.20662181],
       [ 0.22939104,  0.16137405,  0.62867755, ..., -0.05060817,
         0.31964314,  0.2222297 ],
       [ 0.0951618 ,  0.17060103,  0.45820257, ..., -0.0764672 ,
         0.18275926,  0.09100824]], dtype=float32)

In [19]:
import numpy as np
np.save("ag_news_features.npy", features)

In [24]:
labels = df['Class Index'].values
labels

array([3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3,
       3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3,
       3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3,
       3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4,
       4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4,
       4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4,
       4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4,
       4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4,
       4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4,
       4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4,
       4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4,
       4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4,
       4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4,
       4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4,

In [25]:
print(features.shape)
print(labels.shape)


(4000, 768)
(1000,)


In [26]:
features_reshaped = features.reshape((1000, -1))
dataset = np.hstack((features_reshaped, labels.reshape((-1, 1))))
features_reshaped.shape

(1000, 3072)

In [27]:
dataset

array([[ 0.19557555, -0.30943725,  0.34961572, ...,  0.31380805,
        -0.29116049,  3.        ],
       [-0.07465593, -0.07769969, -0.08580375, ...,  0.17737347,
         0.32903737,  3.        ],
       [ 0.25305584, -0.40698659,  0.73172116, ...,  0.15546796,
        -0.33269984,  3.        ],
       ...,
       [ 0.03155316, -0.11947215,  0.50443274, ...,  0.04140321,
        -0.02981426,  4.        ],
       [ 0.09471669,  0.08542237,  0.06571012, ...,  0.09838621,
        -0.26123142,  4.        ],
       [ 0.13106695,  0.05541757,  0.45440722, ...,  0.18275926,
         0.09100824,  4.        ]])

In [28]:
dataset.shape

(1000, 3073)

In [29]:
# Split the data into training and testing sets
train_data, test_data = train_test_split(dataset, test_size=0.2, random_state=42)

# Convert the training and testing sets back into separate feature and label arrays
X_train, y_train = train_data[:, :-1], train_data[:, -1]
X_test, y_test = test_data[:, :-1], test_data[:, -1]

In [30]:
from sklearn.linear_model import LogisticRegression

# Train a logistic regression classifier on the training set
clf = LogisticRegression(max_iter = 1000)
clf.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

In [31]:
score = clf.score(X_test, y_test)
print("Accuracy:", score)

Accuracy: 0.85


In [32]:
y_pred = clf.predict(X_test)

In [34]:
from sklearn.metrics import confusion_matrix, classification_report
cm = confusion_matrix(y_test, y_pred)
cr = classification_report(y_test, y_pred)

In [35]:
print("Confusion Matrix:\n", cm)

Confusion Matrix:
 [[34  2  0  3]
 [ 2 25  0  0]
 [ 0  0 25 12]
 [ 3  0  8 86]]


In [36]:
print("\nClassification Report:\n", cr)


Classification Report:
               precision    recall  f1-score   support

         1.0       0.87      0.87      0.87        39
         2.0       0.93      0.93      0.93        27
         3.0       0.76      0.68      0.71        37
         4.0       0.85      0.89      0.87        97

    accuracy                           0.85       200
   macro avg       0.85      0.84      0.85       200
weighted avg       0.85      0.85      0.85       200

